In [ ]:
import html
import requests

In [ ]:
"""Get all available trivia categories with their IDs."""
url = "https://opentdb.com/api_category.php"
data = requests.get(url).json()

In [ ]:
lines = []
for cat in data['trivia_categories']:
    for cat in data['trivia_categories']:
        lines.append(f"{cat['id']}: {cat['name']}")
    print("\n".join(lines))
    
    
    

In [ ]:
"""Fetch trivia questions.

    Args:
        amount: Number of questions (1-10)
        category: Category ID (use get_categories to see options)
        difficulty: easy, medium, or hard
"""
params = {
        'amount': 10,
        'category': 9,
        'difficulty': 'medium',
        'type': 'multiple',
    }
url =  "https://opentdb.com/api.php"
data = requests.get(url, params=params).json()


In [ ]:
data

In [ ]:
from trivia_tools import TriviaTools
tools = TriviaTools()
print(tools.get_categories())

In [ ]:
from pydantic_ai import Agent
from openai import OpenAI
import dotenv


In [ ]:
dotenv.load_dotenv()
openai = OpenAI()

In [ ]:
instructions = """You are a trivia quizmaster. When asked to play trivia:
1. Use the available tools to fetch trivia questions
2. Ask the player one question at a time with multiple choice options
3. Wait for their answer before moving to the next question
4. When the player answers, explain why the correct answer is correct - add interesting context and facts
5. After all questions, give the final score
"""

trivia_tools = TriviaTools()

agent = Agent(
    'openai:gpt-4o-mini',
    name='agent',
    tools=[trivia_tools.get_categories, trivia_tools.get_questions],
    instructions=instructions,
)

In [ ]:
result = await agent.run("Let's play 3 medium questions from History")

In [ ]:

message_history = result.all_messages()  
print (result.output)
while True:
    player_answer = input("Your answer(or type 'quit' to stop)")
    if  player_answer.strip().lower() == 'quit':
        break      
    result = await agent.run(player_answer,message_history=message_history)
    print (result.output)
    
    
    

In [ ]:
get player answer: inp

In [ ]:
messages = result.all_messages()
for message in result.new_messages():
    for part in message.parts:
        if part.part_kind == "tool-call":
            print(f"Tool:{part.tool_name}")
            print(f"Tool:{part.args}")        


In [1]:
import dotenv
dotenv.load_dotenv()

import os
from logfire.query_client import LogfireQueryClient

read_token = os.getenv('LOGFIRE_TOKEN')
logfire_query_client = LogfireQueryClient(read_token=read_token)

In [9]:
from datetime import datetime, timedelta, timezone

In [12]:
    trace_rows = logfire_query_client.query_json_rows(
        sql="""
        SELECT
            trace_id,
            start_timestamp,
            duration
        FROM records
        WHERE span_name = 'agent run'
           OR span_name LIKE 'trivia_session%'
        ORDER BY start_timestamp DESC
        LIMIT 2
        """,  min_timestamp=datetime.now(timezone.utc) - timedelta(days=7)
    )

In [13]:
trace_rows


{'columns': [{'name': 'trace_id', 'datatype': 'Utf8', 'nullable': False},
  {'name': 'start_timestamp',
   'datatype': {'Timestamp': ['Microsecond', 'UTC']},
   'nullable': False},
  {'name': 'duration', 'datatype': 'Float64', 'nullable': True}],
 'rows': [{'trace_id': '01a18043e6b23f0247fe3e7eab0f96d5',
   'start_timestamp': '2026-10-28T02:08:16.050076Z',
   'duration': 110.8069789}]}

In [14]:
trace_ids = [r['trace_id'] for r in trace_rows['rows']]
trace_ids

['01a18043e6b23f0247fe3e7eab0f96d5']

In [16]:
trace_id = trace_ids[0]

run_row = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        attributes->'pydantic_ai.all_messages' as all_messages,
        attributes->>'gen_ai.usage.input_tokens' as input_tokens,
        attributes->>'gen_ai.usage.output_tokens' as output_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND (span_name = 'agent run' OR span_name LIKE 'invoke_agent %')
    ORDER BY start_timestamp ASC
    """, min_timestamp=datetime.now(timezone.utc) - timedelta(days=7)
)

In [17]:
all_messages = run_row['rows'][0]
all_messages['all_messages'][0]

{'role': 'user',
 'parts': [{'type': 'text',
   'content': "Let's play 5 easy questions from Science & Nature"}]}